In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch

In [2]:
df = pd.DataFrame(
    {
        "area": [120, 180, 150, 210, 105],
        "age": [5, 2, 1, 2, 1],
        "price": [30, 90, 100, 180, 85]
    }
)
print(df.to_string(index=False))

 area  age  price
  120    5     30
  180    2     90
  150    1    100
  210    2    180
  105    1     85


In [3]:
# def normalize(df):
#     # Min-max scaling: (value - min) / (max - min)
#     return (df - df.min()) / (df.max() - df.min())

In [4]:
def normalize(df):
    scaler = StandardScaler()
    return pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

In [5]:
df = normalize(df)
print(df.to_string(index=False))

     area       age     price
-0.858956  1.905159 -1.392212
 0.702782 -0.136083 -0.145455
-0.078087 -0.816497  0.062338
 1.483651 -0.136083  1.724681
-1.249390 -0.816497 -0.249351


In [6]:
X = torch.tensor(df[["area", "age"]].values, dtype=torch.float32)
Y = torch.tensor(df[["price"]].values, dtype=torch.float32)
print(f"{X} \n {Y}")

tensor([[-0.8590,  1.9052],
        [ 0.7028, -0.1361],
        [-0.0781, -0.8165],
        [ 1.4837, -0.1361],
        [-1.2494, -0.8165]]) 
 tensor([[-1.3922],
        [-0.1455],
        [ 0.0623],
        [ 1.7247],
        [-0.2494]])


In [7]:
def stringify_WB(w, b):
    print(w)
    print(b)

    print("Model Predicts:\n")
    print(f"predicted_price = area * {w[0]} + age * {w[1]} + {b} ")

In [8]:
W = torch.rand(size=(2, 1), requires_grad=True)
B = torch.rand(1, requires_grad=True)
print(stringify_WB(W, B))

tensor([[0.5318],
        [0.4766]], requires_grad=True)
tensor([0.7762], requires_grad=True)
Model Predicts:

predicted_price = area * tensor([0.5318], grad_fn=<SelectBackward0>) + age * tensor([0.4766], grad_fn=<SelectBackward0>) + tensor([0.7762], requires_grad=True) 
None


In [9]:
def get_error_df(Y, pred, error):
    # Convert tensors to numpy arrays (if they are tensors) and flatten them
    def to_1d_array(x):
        if hasattr(x, 'detach'):
            return x.detach().cpu().numpy().flatten()
        return x.flatten() # Fallback if they are already NumPy arrays

    # Create the DataFrame
    df = pd.DataFrame({
        'Y': to_1d_array(Y),
        'pred': to_1d_array(pred),
        'loss': to_1d_array(error) # Note: The column is named 'loss' in the output
    })
    
    # Return as a string without the index and formatted to 2 decimal places
    return df.to_string(index=False, float_format="{:.2f}".format)


In [10]:
pred = X @ W + B
error = (Y - pred) ** 2
loss = error.mean()
print(get_error_df(Y, pred, error), "\n", f"Loss: {loss.item()}")

    Y  pred  loss
-1.39  1.23  6.86
-0.15  1.09  1.51
 0.06  0.35  0.08
 1.72  1.50  0.05
-0.25 -0.28  0.00 
 Loss: 1.701641321182251


In [11]:


def stringify_grads(W, B, dW, dB):
    # Convert tensors to numpy for formatting the array string without commas
    w_grad_str = dW.detach().cpu().numpy()
    b_grad_str = dB.detach().cpu().numpy()
    
    lines = [
        f"W: {W}",
        f"B: {B}",
        f"W Gradients: {w_grad_str}, B Gradients: {b_grad_str}",
        "",
        "To reduce loss, Optimizer wants to reduce weights:"
    ]
    
    # Flatten values to make them easy to index
    w_vals = W.detach().flatten().tolist()
    dw_vals = dW.detach().flatten().tolist()
    b_val = B.detach().flatten().item()
    db_val = dB.detach().flatten().item()
    
    # Gradient Descent logic: subtracting a positive grad decreases the value, 
    # subtracting a negative grad increases the value
    w0_action = "decrease" if dw_vals[0] > 0 else "increase"
    w1_action = "decrease" if dw_vals[1] > 0 else "increase"
    b_action = "decrease" if db_val > 0 else "increase"
    
    # Append the formatted explanation strings
    lines.append(f"    W_area from {w_vals[0]:.4f} by {dw_vals[0]:.4f} (i.e. {w0_action})")
    lines.append(f"    W_age from {w_vals[1]:.4f} by {dw_vals[1]:.4f} (i.e. {w1_action})")
    lines.append(f"    Bias from {b_val:.4f} by {db_val:.4f} (i.e. {b_action})")
    
    return "\n".join(lines)

In [12]:
loss.backward()
dW = W.grad
dB = B.grad
print(stringify_grads(W, B, dW, dB))


W: tensor([[0.5318],
        [0.4766]], requires_grad=True)
B: tensor([0.7762], requires_grad=True)
W Gradients: [[-0.68218195]
 [ 1.8582286 ]], B Gradients: [1.5524255]

To reduce loss, Optimizer wants to reduce weights:
    W_area from 0.5318 by -0.6822 (i.e. increase)
    W_age from 0.4766 by 1.8582 (i.e. decrease)
    Bias from 0.7762 by 1.5524 (i.e. decrease)


In [13]:
lr = 0.2
with torch.no_grad():
    W = W - lr * dW
    B = B - lr * dB
print("Updated Weights: \n", stringify_WB(W, B))

tensor([[0.6682],
        [0.1050]])
tensor([0.4657])
Model Predicts:

predicted_price = area * tensor([0.6682]) + age * tensor([0.1050]) + tensor([0.4657]) 
Updated Weights: 
 None


In [ ]:
# # 1. Define your string
# text = "Python is fun. Learning Python is a great way to start programming in Python!"

# # 2. Set up an empty dictionary to hold your counts
# word_counts = {}
# count = 0

# text.lower()
# words = text.split()
# for word in words:
#     if word == words[word+1]:
        

In [1]:
import string
from collections import Counter

def pythonic_word_counter(text):
    clean_text = text.lower().translate(str.maketrans("", "", string.punctuation))

    words = clean_text.split()

    return Counter(words)

text = "Python is fun. Learning Python is a great way to start programming in Python!"

result = pythonic_word_counter(text)

for word, count in result.items():
    print(f"{word}: {count}")

python: 3
is: 2
fun: 1
learning: 1
a: 1
great: 1
way: 1
to: 1
start: 1
programming: 1
in: 1


In [4]:
# Your raw server logs
server_logs = [
    "INFO - User logged in successfully.",
    "ERROR - Database connection timeout.",
    "WARN - CPU usage is exceeding 80%.",
    "INFO - Report generated.",
    "ERROR - Missing API key in request.",
    "INFO - User logged out."
]

# Create an empty list to hold your filtered messages

def server_log(logs):
    error_messages = []

    for log in logs:
        if log.startswith('ERROR'):
            message = log.split('-')[1].strip()
            error_messages.append(message)
    
    return error_messages

server_log(server_logs)


['Database connection timeout.', 'Missing API key in request.']

In [5]:
# Your "API Response" (a list of dictionaries)
purchases = [
    {"item": "Laptop", "category": "Electronics", "price": 1200},
    {"item": "T-Shirt", "category": "Apparel", "price": 25},
    {"item": "Headphones", "category": "Electronics", "price": 150},
    {"item": "Jeans", "category": "Apparel", "price": 60},
    {"item": "Blender", "category": "Kitchen", "price": 40},
    {"item": "Monitor", "category": "Electronics", "price": 300}
]


def api_sum(purchases):
    category_totals = {}
    for purchase in purchases:
        category = purchase["category"]
        price = purchase["price"]

        if category in category_totals:
            category_totals[category] += price
        else:
            category_totals[category] = price

    for cat, total in category_totals.items():
        print(f"{cat}: ${total}")

api_sum(purchases)



Electronics: $1650
Apparel: $85
Kitchen: $40


In [6]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

class UserMetrics(BaseModel):
    user_id: int
    age: int = Field(gt=18, description="User must be older than 18")
    average_spend: float

app = FastAPI()

@app.post("/predict")

async def calculate_risk_score(user_data: UserMetrics):
    risk_score = (user_data.age * user_data.average_spend) / 100

    return {
        "user_id": user_data.user_id,
        "risk_score": risk_score
    }


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

class UserMetrics(BaseModel):
    user_id: int
    age: int = Field(gt=18, description="User must be older than 18")
    average_spend: float

app = FastAPI()

@app.post("/predict")

async def calculate_risk_score(user_data: UserMetrics):
    risk_score = (user_data.age * user_data.average_spend) / 100

    return {
        "user_id": user_data.user_id,
        "risk_score": risk_score
    }


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

class UserMetrics(BaseModel):
    user_id: int
    age: int = Field(gt=18, description="User must be older than 18")
    average_spend: float

app = FastAPI()

@app.post("/predict")

async def calculate_risk_score(user_data: UserMetrics):
    risk_score = (user_data.age * user_data.average_spend) / 100

    return {
        "user_id": user_data.user_id,
        "risk_score": risk_score
    }
